# Step 7a: RAG-Augmented Underwriting Report (LangGraph + Ollama/Qwen + ChromaDB)
## AI-Assisted Health Risk Prediction for Life Insurance Underwriting

A **parallel variant** of Step 7. Same model prediction, same SHAP values, same
three tools - the one thing added is **retrieval-augmented generation**:

```
Model prediction -> SHAP -> [RETRIEVE medical literature] -> LLM agent -> report
```

### Traditional RAG, not agentic RAG
Retrieval runs in **Python, before** the LLM is invoked, and the passages are
injected into the prompt as context. The LLM does **not** get to decide whether
to retrieve. For a medical/underwriting use case every report must be grounded,
so that decision is not left to a model.

| | Step 7 (baseline) | Step 7a (this notebook) |
|---|---|---|
| 3 clinical tools | yes | yes (unchanged) |
| Retrieved literature in prompt | no | yes, always |
| Retrieval decided by | n/a | Python (deterministic) |
| Citations in report | no | yes, `[1]`, `[2]`, … |

Keeping the tools identical means the **only** variable between the two runs is
the retrieved context - a clean A/B comparison for the report and the demo.


### Where things live (all under `Code/`)
| Path | Purpose |
|---|---|
| `Code/rag_utils.py` | load → chunk → embed → ChromaDB → retrieve |
| `Code/knowledge_base/` | drop source PDFs here (see its README) |
| `Code/vectordb/` | ChromaDB persistent store (auto-generated) |

## 7a.1 Prerequisites

1. Ollama installed and running — https://ollama.com/download/windows
2. Chat model: `ollama pull qwen2.5:7b`
3. Embedding model: `ollama pull nomic-embed-text`
4. Packages: `pip install chromadb pypdf`
5. Step 6 must have produced `data/results/shap_explanations.json`

In [1]:
import json
import sys
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent

# rag_utils.py sits next to this notebook in Code/
CODE_DIR = Path.cwd() if Path.cwd().name == 'Code' else Path.cwd() / 'Code'
sys.path.insert(0, str(CODE_DIR.resolve()))
import rag_utils as rag

MODEL_NAME = 'qwen2.5:7b'          # same chat model as Step 7 (fair comparison)
EMBED_MODEL = rag.EMBED_MODEL      # 'nomic-embed-text'

print('Imports OK')
print('Knowledge base :', rag.KB_DIR)
print('Vector store   :', rag.VECTORDB_DIR)

Imports OK
Knowledge base : <PROJECT_ROOT>\Code\knowledge_base
Vector store   : <PROJECT_ROOT>\Code\vectordb


In [2]:
# Connectivity check: BOTH the chat model and the embedding model must be present
import requests

try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    installed = [m['name'] for m in r.json().get('models', [])]
    print('Ollama is running. Installed models:', installed)
    for needed in (MODEL_NAME, EMBED_MODEL):
        stem = needed.split(':')[0]
        ok = any(stem in m for m in installed)
        print(f"  {'OK  ' if ok else 'MISSING'} {needed}"
              + ('' if ok else f'   -> run: ollama pull {needed}'))
except Exception as e:
    print('Could not reach Ollama at localhost:11434.')
    print('Start Ollama, then re-run this cell. Error:', e)

Ollama is running. Installed models: ['nomic-embed-text:latest', 'qwen2.5:7b']
  OK   qwen2.5:7b
  OK   nomic-embed-text


## 7a.2 Build the Vector Index

`rag_utils.build_index()` walks `Code/knowledge_base/` (recursively), extracts
text, splits it into overlapping character chunks, embeds each chunk with
`nomic-embed-text`, and writes them to a ChromaDB collection using **cosine**
distance.

**To add more literature later:** drop the PDFs into `Code/knowledge_base/` and
re-run this cell. The collection is recreated from scratch, so the index always
mirrors the folder exactly - no stale chunks.

In [3]:
# Set REBUILD = False to skip re-embedding and just reuse the existing index.
REBUILD = True

if REBUILD:
    stats = rag.build_index(rebuild=True)
    print('\nIndexed files:')
    for f in stats['files']:
        print('  -', f)
else:
    print('Skipped rebuild.')

info = rag.index_stats()
print('\nIndex status:', 'READY' if info['available'] else 'NOT AVAILABLE')
print('Chunks in collection:', info['chunks'])
print('Sources:', ', '.join(info['sources']) or '(none)')

Loaded 351 page(s) from 50 file(s) in <PROJECT_ROOT>\Code\knowledge_base
Split into 5593 chunks (size=900, overlap=150)
  embedded 5593/5593
Indexed 5593 chunks -> <PROJECT_ROOT>\Code\vectordb

Indexed files:
  - Blood Cholesterol Guideline.pdf
  - Diagnosis and Classification.pdf
  - Hypertension Guideline.pdf
  - WHO obesity and overweight fact sheet.txt
  - cdc_us_life_tables_2021.pdf
  - epmc_albuminuria_risk_1_PMC13235255.txt
  - epmc_albuminuria_risk_1_PMC13292312.txt
  - epmc_albuminuria_risk_2_PMC13231811.txt
  - epmc_albuminuria_risk_2_PMC13274163.txt
  - epmc_blood_pressure_mortality_1_PMC13273731.txt
  - epmc_blood_pressure_mortality_1_PMC13292307.txt
  - epmc_blood_pressure_mortality_2_PMC13288547.txt
  - epmc_blood_pressure_mortality_2_PMC13288904.txt
  - epmc_bmi_obesity_mortality_1_PMC13121583.txt
  - epmc_bmi_obesity_mortality_2_PMC13292310.txt
  - epmc_cancer_survivor_mortality_1_PMC13279761.txt
  - epmc_cancer_survivor_mortality_1_PMC13292245.txt
  - epmc_cancer_survi

In [4]:
# Sanity-check retrieval: does a clinical question pull back sensible passages?
for q in ['what HbA1c value indicates diabetes',
          'congestive heart failure life expectancy underwriting']:
    print('=' * 70)
    print('QUERY:', q)
    for h in rag.retrieve(q, k=2):
        print(f"  [{h['score']:.3f}] {h['source']} p.{h['page']}")
        print('        ', h['text'][:180].replace('\n', ' '), '...')

QUERY: what HbA1c value indicates diabetes


  [0.715] Diagnosis and Classification.pdf p.3
         the diabetes criterion of A1C (two results$6.5% [$48 mmol/mol]) but not FPG (<126 mg/dL [<7.0 mmol/L]), that person should nevertheless be considered to have diabetes. If individua ...
  [0.709] Diagnosis and Classification.pdf p.2
         Diagnosis of Diabetes In the less common clinical scenario where a person has classic hyperglycemic symp- toms (e.g., polyuria, polydipsia, and unex- plained weight loss), measurem ...
QUERY: congestive heart failure life expectancy underwriting
  [0.701] Hypertension Guideline.pdf p.83
         heart failure in elderly persons: influence of left ventricular  systolic function. The Cardiovascular Health Study. Ann Intern  Med. 2002;137:631–9.  S9.2.1-5. Owan TE, Hodge DO,  ...
  [0.692] cdc_us_life_tables_2021.pdf p.61
         by cause of death changes  in life expectancy at birth in the United States from 2020 to 2021. The method partitions changes into component additive  parts and identifie

## 7a.3 Load SHAP Explanations, Clinical Lookups & De-standardize

Identical to Step 7: Step 3 applied a `StandardScaler`, so SHAP `feature_value`
entries are z-scores. We un-transform them so the clinical tools (and the
retrieval queries) work with real-world units.

In [5]:
import joblib
import numpy as np


def find_dir(name):
    for base in [Path('.'), Path('..'), Path('..') / '..']:
        candidate = base / name
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not locate {name}.')


DATA_DIR = find_dir('data')
RESULTS_DIR = DATA_DIR / 'results'
CONFIG_DIR = find_dir('config')
REPORTS_DIR = DATA_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = DATA_DIR / 'model_ready'

with open(RESULTS_DIR / 'shap_explanations.json') as f:
    shap_explanations = json.load(f)

with open(CONFIG_DIR / 'clinical_lookups.json') as f:
    LOOKUPS = json.load(f)
CLINICAL_RANGES = LOOKUPS['clinical_ranges']
RISK_CATEGORIES = LOOKUPS['risk_categories']

# De-standardize feature values back to clinical units
with open(MODEL_DIR / 'preprocessing_metadata.json') as f:
    prep_meta = json.load(f)
preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')
scaler = preprocessor.named_transformers_['num'].named_steps['scale']
scaler_params = {feat: (float(scaler.mean_[i]), float(scaler.scale_[i]))
                 for i, feat in enumerate(prep_meta['continuous_features'])}

for rec in shap_explanations:
    for c in rec['top_contributions']:
        if c['feature'] in scaler_params:
            mean, std = scaler_params[c['feature']]
            c['feature_value_original'] = round(float(c['feature_value'] * std + mean), 2)
        else:
            c['feature_value_original'] = c['feature_value']

print(f'Loaded {len(shap_explanations)} applicant explanations')
print(f'Clinical ranges for {len(CLINICAL_RANGES)} features')
print(f'Risk category bands: {len(RISK_CATEGORIES["bands"])}')

Loaded 8 applicant explanations
Clinical ranges for 12 features
Risk category bands: 5


## 7a.4 The Three Tools

Deliberately **identical** to Step 7 so the A/B comparison isolates the effect of
retrieval. Thresholds still come from `config/clinical_lookups.json`.

In [6]:
@tool
def lookup_clinical_ranges(feature_code: str, value: float) -> str:
    """Interpret a clinical lab or vital-sign value against published reference
    ranges. feature_code is an NHANES code like 'LBXGH' (HbA1c), 'BP_SYS_MEAN'
    (systolic BP), 'BMXBMI' (BMI). value is the measured value in the feature's
    natural units. Returns the matching clinical band (e.g. 'diabetic') and the
    full reference ranges."""
    info = CLINICAL_RANGES.get(feature_code)
    if not info:
        return f'No clinical reference range available for {feature_code}.'
    band_label = 'unknown'
    for b in info['bands']:
        lo = b['min'] if b['min'] is not None else float('-inf')
        hi = b['max'] if b['max'] is not None else float('inf')
        if lo <= value < hi:
            band_label = b['label']
            break
    ranges_txt = '; '.join(
        f"{b['label']}: {b['min']}-{b['max'] if b['max'] is not None else '+'}"
        for b in info['bands'])
    return (f"{info['name']} = {value} {info['unit']} -> category: '{band_label}'. "
            f"Reference ranges [{ranges_txt}]. Source: {info['source']}.")


@tool
def get_risk_category(predicted_risk: float) -> str:
    """Map a predicted mortality probability (0..1) to a simplified life-insurance
    underwriting decision category and recommended action."""
    for b in RISK_CATEGORIES['bands']:
        if b['min'] <= predicted_risk < b['max']:
            return (f"Predicted risk {predicted_risk:.1%} -> category '{b['category']}'. "
                    f"Recommended action: {b['action']}. Rating: {b['table_rating']}.")
    return f'Predicted risk {predicted_risk:.1%} could not be categorized.'


@tool
def explain_shap_contribution(feature_code: str, shap_value: float, direction: str) -> str:
    """Turn a single SHAP contribution into a plain-English risk statement.
    direction is 'increases_risk' or 'decreases_risk'."""
    info = CLINICAL_RANGES.get(feature_code)
    friendly = info['name'] if info else feature_code
    verb = 'increased' if direction == 'increases_risk' else 'decreased'
    strength = ('strongly' if abs(shap_value) > 1.0
                else 'moderately' if abs(shap_value) > 0.3 else 'slightly')
    return (f"{friendly} {strength} {verb} the applicant's mortality risk "
            f"(SHAP impact = {shap_value:+.2f} in log-odds).")


TOOLS = [lookup_clinical_ranges, get_risk_category, explain_shap_contribution]
print('Tools defined:', [t.name for t in TOOLS])

Tools defined: ['lookup_clinical_ranges', 'get_risk_category', 'explain_shap_contribution']


## 7a.5 Build the Agent (RAG-aware system prompt)

`create_react_agent` wiring and same model. The only change is the system
prompt, which now tells the agent that authoritative literature is already in
its context and must be cited by number.

In [7]:
# num_ctx=8192 is REQUIRED for the RAG variant. Ollama's default context is 4096
# tokens, which the retrieval block (~900 tok) plus the system prompt, the three
# tool schemas and the accumulated tool results overflow partway through the ReAct
# loop.
llm = ChatOllama(model=MODEL_NAME, temperature=0.2, num_ctx=8192,
                 client_kwargs={'timeout': 900})

SYSTEM_PROMPT_RAG = '''You are an AI underwriting assistant for a life insurance company.
You receive a mortality-risk prediction (already produced by a machine-learning
model), SHAP explanations of the factors that drove it, and excerpts from
authoritative medical literature that have been retrieved for this applicant.

Your job is to write a clear, professional underwriting summary. You MUST:
- Use get_risk_category once to state the overall underwriting decision.
- Use explain_shap_contribution for the most important factors.
- Use lookup_clinical_ranges to interpret specific clinical values when relevant.
- Ground your clinical reasoning in the RETRIEVED LITERATURE supplied in the
  prompt, and cite it inline by its bracket number, e.g. [1], [2].
- Only cite a number that actually appears in the retrieved literature block.
- If the retrieved literature does not cover a factor, say so plainly instead of
  inventing support for it.
- Never invent numbers. Only use values provided or returned by tools.
- NEVER change or re-estimate the mortality prediction yourself; you only explain it.

Write the final report in this structure:
1. Underwriting Decision (category + action)
2. Key Risk Factors (bulleted, most important first, with citations)
3. Protective Factors (if any)
4. Clinical Context from Literature (2-4 bullets, each with a citation)
5. Summary statement (2-3 sentences).'''

agent_rag = create_react_agent(llm, tools=TOOLS)
print('RAG agent built with chat model:', MODEL_NAME)
print('Embedding model              :', EMBED_MODEL)

RAG agent built with chat model: qwen2.5:7b
Embedding model              : nomic-embed-text


<TEMP>\ipykernel_5876\451276095.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_rag = create_react_agent(llm, tools=TOOLS)


## 7a.6 Build the Augmented Prompt

Two steps. `build_applicant_prompt`. Then
`build_rag_prompt` retrieves and appends the literature block.

**Retrieval strategy (multi-query):** one semantic query per top SHAP driver -
the NHANES code is translated into clinical language by
`rag_utils.feature_topic()`, plus one general underwriting query. Results are
merged, de-duplicated, ranked by cosine similarity, and the top-k are injected.

In [8]:
TOP_K = 4   # passages injected per applicant


def build_applicant_prompt(rec):
    """Applicant data block """
    lines = [f"Applicant ID: {rec['applicant_id']}",
             f"Predicted mortality risk: {rec['predicted_risk']:.4f}",
             '',
             'Top contributing factors (from SHAP):']
    for c in rec['top_contributions']:
        orig_val = c.get('feature_value_original', c['feature_value'])
        lines.append(f"  - {c['feature']}: SHAP={c['shap_value']:+.3f}, "
                     f"value={orig_val}, {c['direction']}")
    return '\n'.join(lines)


def build_rag_prompt(rec, k=TOP_K):
    """Applicant block + retrieved literature. Returns (prompt, passages)."""
    factors = [{'feature': c['feature']} for c in rec['top_contributions']]
    block, passages = rag.build_rag_block(factors, k=k)
    prompt = (build_applicant_prompt(rec) + '\n\n' + block + '\n\n'
              + 'Write the underwriting summary. Use your tools to interpret the '
                'risk category and the key factors (pass the original-scale value '
                'shown above to lookup_clinical_ranges). Support your clinical '
                'reasoning with the retrieved literature and cite it as [1], [2], etc.')
    return prompt, passages


example = shap_explanations[0]
prompt_rag, passages = build_rag_prompt(example)

print('Retrieval queries used:')
for c in example['top_contributions'][:4]:
    print(f"  {c['feature']:22s} -> {rag.feature_topic(c['feature'])}")
print(f'\nRetrieved {len(passages)} passages:')
for i, p in enumerate(passages, 1):
    print(f"  [{i}] {p['source']} p.{p['page']}  similarity={p['score']:.3f}")
print('\n' + '=' * 70)
print(prompt_rag)

Retrieval queries used:
  AGE                    -> advancing age and all-cause mortality risk
  MCQ160B_1.0            -> congestive heart failure prognosis and mortality
  LBXSATSI               -> alanine aminotransferase ALT liver enzyme elevation
  LBXSCR                 -> Serum Creatinine reference range and mortality risk

Retrieved 4 passages:
  [1] epmc_heart_failure_prognosis_1_PMC13276809.txt p.0  similarity=0.769
  [2] epmc_blood_pressure_mortality_1_PMC13292307.txt p.0  similarity=0.733
  [3] Hypertension Guideline.pdf p.84  similarity=0.730
  [4] cdc_us_life_tables_2021.pdf p.5  similarity=0.719

Applicant ID: 118
Predicted mortality risk: 0.9939

Top contributing factors (from SHAP):
  - AGE: SHAP=+2.232, value=80.0, increases_risk
  - MCQ160B_1.0: SHAP=+0.418, value=1.0, increases_risk
  - LBXSATSI: SHAP=+0.287, value=2.71, increases_risk
  - LBXSCR: SHAP=+0.278, value=1.1, increases_risk
  - MCQ160F_1.0: SHAP=+0.229, value=1.0, increases_risk
  - LBDHDD: SHAP=+0.211, 

## 7a.7 Generate a RAG Report for One Applicant

In [9]:
result = agent_rag.invoke({'messages': [
    SystemMessage(content=SYSTEM_PROMPT_RAG),
    HumanMessage(content=prompt_rag),
]})

final_report = result['messages'][-1].content
print('=' * 70)
print(f'RAG UNDERWRITING REPORT - Applicant {example["applicant_id"]}')
print(f'(Actual outcome: {"Deceased" if example["actual_outcome"] == 1 else "Alive"})')
print('=' * 70)
print(final_report)
print('\n' + '-' * 70)
print('SOURCES SUPPLIED TO THE MODEL')
for i, p in enumerate(passages, 1):
    print(f"  [{i}] {p['source']}, page {p['page']} (similarity {p['score']:.3f})")

RAG UNDERWRITING REPORT - Applicant 118
(Actual outcome: Deceased)
1. Underwriting Decision (category + action)
   The predicted mortality risk of 99.4% falls into the 'Decline' category, recommending a decline or postpone decision with referral for medical review. Rating: Decline.

2. Key Risk Factors:
   - Age: 80 years old [3]
     [3] Hypertension Guideline.pdf, p.84 (similarity 0.73) states that age is a significant risk factor for heart failure and mortality.
   - MCQ160B_1.0: Value = 1.0
     This factor likely relates to the presence of comorbidities or medical conditions [2].
   - Serum Creatinine (LBXSCR): 1.1 mg/dL, considered 'normal' by clinical reference ranges.
     The literature suggests that higher creatinine levels are often associated with a lower predicted risk due to hemodilution, reduced muscle mass, impaired creatinine generation, or diminished physiological reserve [1].

3. Protective Factors:
   - No significant protective factors were identified in the provid

In [10]:
# Tool-call trace: the agent still reasons and picks tools, it just now has
# retrieved literature sitting in its context from the very first turn.
print('Agent message trace:')
for m in result['messages']:
    role = type(m).__name__.replace('Message', '')
    if getattr(m, 'tool_calls', None):
        for tc in m.tool_calls:
            print(f'  [{role}] TOOL CALL: {tc["name"]}({tc["args"]})')
    elif m.content:
        preview = m.content[:120].replace(chr(10), ' ')
        print(f'  [{role}] {preview}...')

Agent message trace:
  [System] You are an AI underwriting assistant for a life insurance company. You receive a mortality-risk prediction (already prod...
  [Human] Applicant ID: 118 Predicted mortality risk: 0.9939  Top contributing factors (from SHAP):   - AGE: SHAP=+2.232, value=80...
  [AI] TOOL CALL: get_risk_category({'predicted_risk': 0.9939})
  [AI] TOOL CALL: lookup_clinical_ranges({'feature_code': 'LBXSCR', 'value': 1.1})
  [Tool] Predicted risk 99.4% -> category 'Decline'. Recommended action: Decline or postpone; refer for medical review. Rating: D...
  [Tool] Serum Creatinine = 1.1 mg/dL -> category: 'normal'. Reference ranges [normal: 0-1.3; mildly elevated: 1.3-2.0; elevated ...
  [AI] 1. Underwriting Decision (category + action)    The predicted mortality risk of 99.4% falls into the 'Decline' category,...


## 7a.8 Generate RAG Reports for All Sampled Applicants

Saved separately from Step 7 so both sets survive side by side.

In [11]:
import time

rag_reports = []
for rec in shap_explanations:
    t0 = time.perf_counter()
    p, passages_i = build_rag_prompt(rec)
    res = agent_rag.invoke({'messages': [
        SystemMessage(content=SYSTEM_PROMPT_RAG),
        HumanMessage(content=p),
    ]})
    elapsed = time.perf_counter() - t0
    rag_reports.append({
        'applicant_id': rec['applicant_id'],
        'predicted_risk': rec['predicted_risk'],
        'actual_outcome': rec['actual_outcome'],
        'approach': 'tools+rag',
        'report': res['messages'][-1].content,
        'retrieved': [{'n': i, 'source': q['source'], 'page': q['page'],
                       'score': q['score']}
                      for i, q in enumerate(passages_i, 1)],
        'seconds': round(elapsed, 1),
    })
    print(f"Done applicant {rec['applicant_id']} "
          f"(risk {rec['predicted_risk']:.2f}, {len(passages_i)} passages, {elapsed:.1f}s)")

print(f'\nGenerated {len(rag_reports)} RAG reports.')

Done applicant 118 (risk 0.99, 4 passages, 51.5s)
Done applicant 12662 (risk 0.00, 4 passages, 68.1s)
Done applicant 6339 (risk 0.11, 4 passages, 73.5s)
Done applicant 8673 (risk 0.11, 4 passages, 130.9s)
Done applicant 10891 (risk 0.11, 4 passages, 113.0s)
Done applicant 17300 (risk 0.11, 4 passages, 160.1s)
Done applicant 1176 (risk 0.11, 4 passages, 65.8s)
Done applicant 16988 (risk 0.11, 4 passages, 129.2s)

Generated 8 RAG reports.


In [12]:
# Save (JSON + readable markdown) alongside the Step 7 outputs
json_path = RESULTS_DIR / 'underwriting_reports_rag.json'
with open(json_path, 'w') as f:
    json.dump(rag_reports, f, indent=2)

md_lines = ['# AI-Generated Underwriting Reports (RAG-Augmented)\n',
            f'Model: XGBoost mortality prediction + SHAP + {MODEL_NAME} '
            f'(LangGraph agent) + traditional RAG over ChromaDB '
            f'(embeddings: {EMBED_MODEL})\n']
for r in rag_reports:
    outcome = 'Deceased' if r['actual_outcome'] == 1 else 'Alive'
    md_lines.append(f"\n---\n\n## Applicant {r['applicant_id']} "
                    f"(predicted risk {r['predicted_risk']:.1%}, actual: {outcome})\n")
    md_lines.append(r['report'])
    md_lines.append('\n**Retrieved sources**\n')
    for q in r['retrieved']:
        md_lines.append(f"- [{q['n']}] {q['source']}, page {q['page']} "
                        f"(similarity {q['score']:.2f})")

md_path = REPORTS_DIR / 'underwriting_reports_rag.md'
md_path.write_text('\n'.join(md_lines), encoding='utf-8')
print(f'Saved:\n  - {json_path}\n  - {md_path}')

Saved:
  - <PROJECT_ROOT>\data\results\underwriting_reports_rag.json
  - <PROJECT_ROOT>\data\reports\underwriting_reports_rag.md


## 7a.9 A/B Comparison: Tools-only (Step 7) vs Tools+RAG (Step 7a)

Requires `data/results/underwriting_reports.json` from Step 7. Metrics are
deliberately simple and mechanical — the qualitative judgement belongs in the
written report.

In [13]:
import re
import pandas as pd

baseline_path = RESULTS_DIR / 'underwriting_reports.json'
if not baseline_path.exists():
    print(f'{baseline_path.name} not found - run 07_llm_underwriting_agent.ipynb '
          f'first to enable the comparison.')
    comparison = None
else:
    with open(baseline_path) as f:
        baseline_reports = {r['applicant_id']: r for r in json.load(f)}

    def n_citations(text):
        return len(set(re.findall(r'\[(\d+)\]', text or '')))

    def mentions_top_driver(text, rec):
        """Does the narrative reference the single largest SHAP driver?

        Uses rag_utils.mentions_feature, which applies the SAME synonym-expanded
        matcher as notebook 08 (e.g. 'BMXWAIST' also matches 'waist'). Both arms
        of the A/B are scored with it, so the comparison stays fair and the
        reported figure is consistent across the whole project.
        """
        top = rec['top_contributions'][0]['feature']
        return rag.mentions_feature(text, top)

    rows = []
    shap_by_id = {r['applicant_id']: r for r in shap_explanations}
    for r in rag_reports:
        aid = r['applicant_id']
        base = baseline_reports.get(aid)
        rec = shap_by_id[aid]
        rows.append({
            'applicant_id': aid,
            'predicted_risk': round(r['predicted_risk'], 4),
            'actual_outcome': r['actual_outcome'],
            'words_no_rag': len((base or {}).get('report', '').split()),
            'words_rag': len(r['report'].split()),
            'citations_no_rag': n_citations((base or {}).get('report', '')),
            'citations_rag': n_citations(r['report']),
            'top_driver_no_rag': mentions_top_driver((base or {}).get('report', ''), rec),
            'top_driver_rag': mentions_top_driver(r['report'], rec),
            'passages_retrieved': len(r['retrieved']),
            'mean_similarity': round(
                sum(q['score'] for q in r['retrieved']) / max(len(r['retrieved']), 1), 3),
        })

    comparison = pd.DataFrame(rows)
    csv_path = RESULTS_DIR / 'rag_vs_norag_comparison.csv'
    comparison.to_csv(csv_path, index=False)
    print(f'Saved {csv_path}\n')
    display(comparison)
    print('\nAggregate:')
    print(f"  mean words      no-RAG {comparison['words_no_rag'].mean():6.1f}  "
          f"RAG {comparison['words_rag'].mean():6.1f}")
    print(f"  mean citations  no-RAG {comparison['citations_no_rag'].mean():6.2f}  "
          f"RAG {comparison['citations_rag'].mean():6.2f}")
    print(f"  top driver named no-RAG {int(comparison['top_driver_no_rag'].sum())}/"
          f"{len(comparison)}   RAG {int(comparison['top_driver_rag'].sum())}/{len(comparison)}")

Saved <PROJECT_ROOT>\data\results\rag_vs_norag_comparison.csv



,applicant_id,predicted_risk,actual_outcome,words_no_rag,words_rag,citations_no_rag,citations_rag,top_driver_no_rag,top_driver_rag,passages_retrieved,mean_similarity
0,118,0.9939,1,142,222,0,3,True,True,4,0.738
1,12662,0.0020,0,165,251,0,3,True,True,4,0.704
2,6339,0.1083,0,171,187,0,2,True,True,4,0.714
3,8673,0.1083,0,198,235,0,3,True,True,4,0.701
4,10891,0.1086,0,148,215,0,4,True,True,4,0.714
5,17300,0.1087,0,165,280,0,3,True,True,4,0.731
6,1176,0.1089,0,225,224,0,4,True,True,4,0.717
7,16988,0.1089,0,216,259,0,2,True,True,4,0.703



Aggregate:
  mean words      no-RAG  178.8  RAG  234.1
  mean citations  no-RAG   0.00  RAG   3.00
  top driver named no-RAG 8/8   RAG 8/8


In [14]:
# Side-by-side read of one applicant, for the report appendix / slides
aid = rag_reports[0]['applicant_id']
if baseline_path.exists():
    print('#' * 70)
    print(f'APPLICANT {aid} - TOOLS ONLY (Step 7)')
    print('#' * 70)
    print(baseline_reports[aid]['report'])
print('\n' + '#' * 70)
print(f'APPLICANT {aid} - TOOLS + RAG (Step 7a)')
print('#' * 70)
print(rag_reports[0]['report'])

######################################################################
APPLICANT 118 - TOOLS ONLY (Step 7)
######################################################################
1. Underwriting Decision (category + action)
   - Category: Decline
   - Action: Decline or postpone; refer for medical review

2. Key Risk Factors:
   - Age: 80 years, significantly increases risk (+2.232 log-odds impact).
   - MCQ160B_1.0 and MCQ160F_1.0 (unknown specific codes): Both increase risk by +0.418 and +0.229 log-odds, respectively.
   - Low HDL Cholesterol: 3.3 mg/dL, categorized as 'low (higher risk)' (-0.211 log-odds impact).
   - Elevated Serum Creatinine: 1.1 mg/dL, within the normal range but slightly elevated for renal function (+0.278 log-odds impact).

3. Protective Factors:
   - None identified in this case.

4. Summary statement:
The applicant's overall risk is extremely high, primarily driven by advanced age and certain lifestyle or health factors indicated by the SHAP analysis. The low 

## Wrap-up & Report Notes

**What Step 7a adds over Step 7:**
- **Grounding by construction.** Retrieval is deterministic and always runs, so
  every report is written against source literature rather than model memory.
- **Traceability.** Each report ships with the exact passages, filenames, page
  numbers and similarity scores that were supplied — auditable, which matters in
  a regulated domain.
- **Extensibility.** Adding literature is a file copy plus one cell re-run; no
  code or prompt changes.

**Design trade-off worth stating in the paper:** making retrieval a *tool* would
let the agent skip it. That is cheaper in tokens but unacceptable when every
underwriting narrative must be evidence-backed. Traditional RAG trades some
efficiency for a guarantee.

**Limitations:**
- Report quality is bounded by what is in `Code/knowledge_base/`. The shipped
  seed file is a placeholder; real guideline PDFs should replace it.
- Retrieval quality depends on the NHANES-code → clinical-phrase mapping in
  `rag_utils.feature_topic()`; unmapped codes fall back to a generic query.
- A 7B local model still occasionally mis-cites. The citation count in 7a.9 is a
  coverage signal, not a correctness proof.
- Human-in-the-loop review remains mandatory.

**Next:** the demo UI (`demo/app.py`) exposes an approach selector so tools-only
and tools+RAG can be run and compared live.